In [8]:
from datetime import datetime
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\work\ner")
LLM_INPUT_DIR = PROJECT_ROOT / "data" / "filter" / "llm_input_data" / "20261110_175720"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if not LLM_INPUT_DIR.is_dir():
    raise FileNotFoundError(f"LLM input directory not found: {LLM_INPUT_DIR}")

input_files = sorted(LLM_INPUT_DIR.glob("*.pkl"))
if not input_files:
    raise FileNotFoundError(f"No .pkl files found: {LLM_INPUT_DIR}")

print(f"input directory: {LLM_INPUT_DIR}")
print(f"input files: {len(input_files)}")
[path.name for path in input_files]

input directory: C:\work\ner\data\filter\llm_input_data\20261110_175720
input files: 18


['강원일보.pkl',
 '경기일보.pkl',
 '경향신문.pkl',
 '노컷뉴스.pkl',
 '뉴스1.pkl',
 '뉴스핌.pkl',
 '뉴시스.pkl',
 '디지털타임스.pkl',
 '마이데일리.pkl',
 '부산일보.pkl',
 '서울신문.pkl',
 '세계일보.pkl',
 '스포츠경향.pkl',
 '스포츠투데이.pkl',
 '아시아투데이.pkl',
 '이투데이.pkl',
 '일간투데이.pkl',
 '충청일보.pkl']

In [9]:
input_files = input_files[1:]
input_files

[WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/경기일보.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/경향신문.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/노컷뉴스.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/뉴스1.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/뉴스핌.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/뉴시스.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/디지털타임스.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/마이데일리.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/부산일보.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/서울신문.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/세계일보.pkl'),
 WindowsPath('C:/work/ner/data/filter/llm_input_data/20261110_175720/스포츠경향.pkl'),
 WindowsPath('C:/work/ner/d

In [14]:
import importlib
import src.ner_review.response as response_module
response_module = importlib.reload(response_module)

API_PROVIDER = "OPENAI"
if API_PROVIDER:
    response_module.API_PROVIDER = API_PROVIDER

MODEL_NAME_OVERRIDE = "gpt-5.1"  # claude-haiku-5-5 노트북에서 직접 사용할 모델명. 환경변수 사용 시 None
if MODEL_NAME_OVERRIDE:
    response_module.MODEL_NAME = MODEL_NAME_OVERRIDE

from src.ner_review.response import (
    API_PROVIDER,
    MODEL_NAME,
    PROMPT_DIR,
    PROMPT_VERSION,
    configure_logging,
    create_client,
    create_model_caller,
    review_file,
)
from src.ner_review.prompt import build_system_prompt
from src.ner_review.result import write_jsonl
from src.ner_review.storage import save_run_parquet

run_id = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
output_root = PROJECT_ROOT / "outputs" / LLM_INPUT_DIR.name / run_id / "reports"
output_root.mkdir(parents=True, exist_ok=True)

run_metadata = {
    "run_id": run_id,
    "provider": API_PROVIDER,
    "model": MODEL_NAME,
    "prompt_version": PROMPT_VERSION,
    "data_version": LLM_INPUT_DIR.name,
    "load_dt": datetime.now().date().isoformat(),
}

with (output_root.parent / "run.jsonl").open("w", encoding="utf-8") as run_file:
    write_jsonl(run_file, run_metadata)
save_run_parquet(output_root.parent / "ner_runs.parquet", run_metadata)

configure_logging()
system_prompt = build_system_prompt(PROMPT_DIR)
call_model = create_model_caller(create_client())

print(f"run_id: {run_id}")
print(f"provider/model: {API_PROVIDER} / {MODEL_NAME}")
print(f"output directory: {output_root}")

run_id: run_20261111_201907
provider/model: OPENAI / gpt-5.1
output directory: C:\work\ner\outputs\20261110_175720\run_20261111_201907\reports


In [15]:
for input_file in input_files:
    print(f"processing: {input_file.name}")
    review_file(
        input_file=input_file,
        output_root=output_root,
        run_id=run_id,
        data_version=LLM_INPUT_DIR.name,
        call_model=call_model,
        system_prompt=system_prompt,
    )

print("response completed")
print(f"reports: {output_root}")

2026-11-11 20:19:16,537 [INFO] 프롬프트 적용완료: 문서=NLRW2600000002.5


processing: 경기일보.pkl


2026-11-11 20:19:22,178 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:19:22,539 [WARNING] 검수 결과 행 제외 1건: ['target_span does not match entity_id: NLRW2600000002.5.2.2.E2']
2026-11-11 20:19:22,558 [INFO] 검수완료: NLRW2600000002.5
2026-11-11 20:19:22,558 [INFO] 프롬프트 적용완료: 문서=NLRW2600000002.10
2026-11-11 20:19:32,494 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:19:32,694 [INFO] 검수완료: NLRW2600000002.10
2026-11-11 20:19:32,726 [INFO] 결과저장완료: C:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\경기일보
2026-11-11 20:19:32,733 [INFO] 프롬프트 적용완료: 문서=NWRW2600000001.4


processing: 경향신문.pkl


2026-11-11 20:19:45,838 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:19:45,845 [INFO] 검수완료: NWRW2600000001.4
2026-11-11 20:19:45,847 [INFO] 프롬프트 적용완료: 문서=NWRW2600000001.2
2026-11-11 20:19:48,609 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:19:49,206 [INFO] 검수완료: NWRW2600000001.2
2026-11-11 20:19:49,208 [INFO] 프롬프트 적용완료: 문서=NWRW2600000001.5
2026-11-11 20:19:52,897 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:19:53,441 [INFO] 검수완료: NWRW2600000001.5
2026-11-11 20:19:53,443 [INFO] 프롬프트 적용완료: 문서=NWRW2600000001.10
2026-11-11 20:19:58,185 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:19:58,739 [INFO] 검수완료: NWRW2600000001.10
2026-11-11 20:19:58,740 [INFO] 프롬프트 적용완료: 문서=NWRW2600000001.20
2026-11-11 20:20:03,522 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20

processing: 노컷뉴스.pkl


... (26 outputs left out)

2026-11-11 20:22:57,170 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:22:57,510 [INFO] 검수완료: NWRW2600000005.7
2026-11-11 20:22:57,512 [INFO] 프롬프트 적용완료: 문서=NWRW2600000005.13
2026-11-11 20:23:02,177 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:23:02,713 [INFO] 검수완료: NWRW2600000005.13
2026-11-11 20:23:02,713 [INFO] 프롬프트 적용완료: 문서=NWRW2600000005.15
2026-11-11 20:23:06,728 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:23:07,326 [INFO] 검수완료: NWRW2600000005.15
2026-11-11 20:23:07,339 [INFO] 결과저장완료: C:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\일간투데이
2026-11-11 20:23:07,343 [INFO] 프롬프트 적용완료: 문서=NLRW2600000004.4


processing: 충청일보.pkl


2026-11-11 20:23:14,743 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:23:14,934 [INFO] 검수완료: NLRW2600000004.4
2026-11-11 20:23:14,935 [INFO] 프롬프트 적용완료: 문서=NLRW2600000004.19
2026-11-11 20:23:18,178 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:23:18,178 [INFO] 검수완료: NLRW2600000004.19
2026-11-11 20:23:18,178 [INFO] 프롬프트 적용완료: 문서=NLRW2600000004.24
2026-11-11 20:23:21,935 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:23:22,745 [INFO] 검수완료: NLRW2600000004.24
2026-11-11 20:23:22,745 [INFO] 프롬프트 적용완료: 문서=NLRW2600000004.26
2026-11-11 20:23:33,194 [INFO] HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-11-11 20:23:33,788 [INFO] 검수완료: NLRW2600000004.26
2026-11-11 20:23:33,797 [INFO] 결과저장완료: C:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\충청일보


response completed
reports: C:\work\ner\outputs\20261110_175720\run_20261111_201907\reports


In [16]:
import pandas as pd
from src.ner_review.storage import REVIEW_COLUMNS

parquet_paths = sorted(output_root.glob("*/ner_reviews.parquet"))
if len(parquet_paths) != len(input_files):
    raise RuntimeError(
        f"review parquet count mismatch: {len(parquet_paths)} / {len(input_files)}"
    )

review_frame = pd.concat(
    [pd.read_parquet(path) for path in parquet_paths],
    ignore_index=True,
)

missing_columns = [column for column in REVIEW_COLUMNS if column not in review_frame.columns]
if missing_columns:
    raise RuntimeError(f"review schema columns missing: {missing_columns}")

print(f"review rows: {len(review_frame):,}")
print(f"review schema columns: {list(review_frame.columns)}")
print(f"reviewed rows: {(review_frame['llm_review_status'] == 'reviewed').sum():,}")
print(f"manual review rows: {review_frame['review_needed'].fillna(False).sum():,}")

review_csv_path = output_root.parent / "ner_reviews.csv"
csv_columns = [
    "model", "data_version", "press", "doc_id",
    "sentence_id", "entity_id", "form", "review_status",
    "error_lv1", "error_lv2", "reason", "target_span",
    "corrected_span", "correctedd_tag", "review_needed", "load_dt",
]
review_csv_frame = review_frame.copy()
review_csv_frame["model"] = MODEL_NAME
review_csv_frame["data_version"] = LLM_INPUT_DIR.name
review_csv_frame["form"] = pd.NA
review_csv_frame["correctedd_tag"] = pd.NA
review_csv_frame = review_csv_frame.reindex(columns=csv_columns)
review_csv_frame.to_csv(review_csv_path, index=False, encoding="utf-8-sig")
print(f"review csv: {review_csv_path}")
display(review_frame[REVIEW_COLUMNS].head(20))

review rows: 2,120
review schema columns: ['run_id', 'doc_id', 'sentence_id', 'entity_id', 'target_span', 'target_type', 'corrected_span', 'error_lv1', 'error_lv2', 'reason', 'review_needed', 'llm_review_status', 'review_status', 'load_dt', 'press']
reviewed rows: 2,120
manual review rows: 118
review csv: C:\work\ner\outputs\20261110_175720\run_20261111_201907\ner_reviews.csv


,run_id,doc_id,sentence_id,entity_id,target_span,target_type,corrected_span,error_lv1,error_lv2,reason,review_needed,llm_review_status,review_status,load_dt,press
0,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.1.1,NLRW2600000002.5.1.1.E1,<김정헌: PS>,PS,NaN,NaN,None,NaN,False,reviewed,normal,2026-11-11,경기일보
1,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.1.1,NLRW2600000002.5.1.1.E2,<인천: LC>,LC,NaN,NaN,None,NaN,False,reviewed,normal,2026-11-11,경기일보
2,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.2.1,NLRW2600000002.5.2.1.E1,<김정헌: PS>,PS,NaN,NaN,None,NaN,False,reviewed,normal,2026-11-11,경기일보
3,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.2.1,NLRW2600000002.5.2.1.E2,<인천: LC>,LC,NaN,NaN,None,NaN,False,reviewed,normal,2026-11-11,경기일보
4,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.2.1,NLRW2600000002.5.2.1.E3,<영종1동: LC>,LC,NaN,NaN,None,NaN,False,reviewed,normal,2026-11-11,경기일보
5,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.2.2,NLRW2600000002.5.2.2.E1,<10일: DT>,DT,NaN,NaN,None,NaN,False,reviewed,normal,2026-11-11,경기일보
6,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.2.2,NLRW2600000002.5.2.2.E2,<중구: LC>,LC,NaN,NaN,None,[자동검증 실패] LLM 검수 결과를 원본 개체와 안전하게 매칭하지 못함,True,reviewed,error,2026-11-11,경기일보
7,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.2.2,NLRW2600000002.5.2.2.E3,<보건소: OG>,OG,NaN,NaN,None,NaN,False,reviewed,normal,2026-11-11,경기일보
8,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.2.3,NaN,NaN,NaN,NaN,NaN,None,NaN,False,reviewed,normal,2026-11-11,경기일보
9,run_20261111_201907,NLRW2600000002.5,NLRW2600000002.5.2.4,NLRW2600000002.5.2.4.E1,<영종: LC>,LC,NaN,NaN,None,NaN,False,reviewed,normal,2026-11-11,경기일보
